# Amazon Bedrock AgentCore Runtime 및 AgentCore Memory Agent

## 개요

이 튜토리얼에서는 AgentCore Runtime과 AgentCore Memory를 사용하여 첫 번째 Memory 지원 Agent를 만드는 방법을 살펴봅니다. 세션 내에서 이전 상호작용을 기억하는 간단한 "Hello World" 대화형 Agent를 구축합니다.

### 튜토리얼 세부 정보


| 정보         | 세부 정보                                                          |
|:--------------------|:-----------------------------------------------------------------|
| 튜토리얼 유형       | Hello World / 소개                                       |
| Agent 유형          | 단일 대화형 Agent                                      |
| Agentic Framework   | Strands Agents                                                   |
| LLM 모델           | Anthropic Claude Haiku 4.5                                      |
| 주요 기능        | AgentCore Runtime, Memory 통합                            |
| 예제 난이도  | 초급                                                         |
| 사용 SDK            | boto3, bedrock-agentcore, bedrock-agentcore-starter-toolkit      |

### 학습 내용

이 튜토리얼에서는 다음 내용을 학습합니다.
1. Agent용 Memory 리소스를 생성하는 방법
2. 자동 대화 유지를 위해 AgentCoreMemorySessionManager를 사용하는 방법
3. Agent를 AgentCore Runtime에 배포하는 방법
4. 세션 관리 기능으로 Agent를 테스트하는 방법


### 아키텍처

이 Hello World 예제는 Memory가 통합되어 AgentCore Runtime에 배포된 간단한 대화형 Agent를 보여 줍니다.

<div style="text-align:left">
    <img src="RuntimeMemoryIntegration.png" width="90%"/>
</div>


## 0. 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10 이상
* AWS 자격 증명 구성
* Amazon Bedrock 모델 액세스(Claude Haiku 4.5)
* Amazon Bedrock AgentCore SDK

먼저 필요한 라이브러리를 설치합니다.

In [ ]:
!pip3 install -qr requirements.txt

### 환경 설정

필요한 라이브러리를 가져오고 환경을 구성합니다.

In [ ]:
# 가져오기
import os
import boto3
import uuid
import logging
from bedrock_agentcore.memory import MemoryClient, MemorySessionManager

# 구성
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("runtime-memory-agent")
REGION = os.getenv("AWS_REGION", "us-west-2")  # Agent용 AWS 리전
memory_client = MemoryClient(region_name=REGION)

## 1. Memory 리소스 생성

이 섹션에서는 Agent가 대화 기록을 저장할 Memory 리소스를 생성합니다. Memory를 사용하면 Agent가 과거 상호작용을 기억하고 컨텍스트를 유지하여 시간이 지나도 더 일관된 응답을 제공할 수 있습니다.

이 예제에서는 추가 장기 strategy 없이 간단한 단기 Memory 리소스를 생성합니다. Memory는 모든 대화 메시지를 저장하여 AgentCore Runtime에서 세션이 종료된 후에도 세션을 이어 갈 때 Agent가 이전 상호작용을 기억하도록 합니다.

In [ ]:
from botocore.exceptions import ClientError

# 이 리소스의 고유 식별자 생성
unique_id = str(uuid.uuid4())[:8]
memory_name = f"RuntimeMemoryAgent_{unique_id}"

try:
    # strategy 없이 Memory 리소스 생성(단기 메모리만 사용)
    memory = memory_client.create_memory_and_wait(
        name=memory_name,
        strategies=[],  # 단기 메모리에는 strategy를 사용하지 않음
        description="Short-term memory for AgentCore Runtime agent",
        event_expiry_days=7,  # 단기 메모리 보존 기간
    )
    memory_id = memory["id"]
    logger.info(f"✅ Created memory: {memory_id}")
except ClientError as e:
    logger.info(f"❌ ERROR: {e}")
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(e):
        # Memory가 이미 존재하면 ID 검색
        memories = memory_client.list_memories()
        memory_id = next((m["id"] for m in memories if m["id"].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Memory 생성 중 발생한 오류 표시
    logger.error(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()
    # 오류 발생 시 정리 - 일부 생성된 Memory 삭제
    if "memory_id" in locals() and memory_id:
        try:
            memory_client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")

## 2. Memory 지원 Agent 생성

이 섹션에서는 기본 제공 AgentCoreMemorySessionManager를 사용하여 Memory가 통합된 Strands Agents framework 기반 Agent를 구축합니다.

> **Memory가 중요한 이유**: AgentCore Runtime의 세션은 일정 시간이 지나면 만료되어 대화 컨텍스트가 삭제됩니다. 대화를 Memory에 저장하면 세션 간에 이전 정보가 유지되므로 오랜 시간이 지난 뒤에도 사용자에게 자연스러운 경험을 제공할 수 있습니다.

### Agent 기능

Agent는 다음 작업을 수행합니다.
1. 각 사용자 및 Assistant 메시지를 Memory에 자동 저장
2. 기존 세션을 이어 갈 때 과거 대화 기록 검색
3. 동일한 사용자와의 여러 상호작용에서 컨텍스트 유지

### 구현의 주요 구성 요소

#### 1. AgentCoreMemorySessionManager(권장)
다음 작업을 자동으로 처리하는 기본 제공 Strands 통합 기능입니다.
- 각 사용자 및 Assistant 메시지를 Memory에 저장
- 기존 세션을 이어 갈 때 과거 대화 기록 검색
- session 및 actor 컨텍스트 관리

#### 2. Agent 초기화
`initialize_agent` 함수는 다음을 수행합니다.
- memory_id, session_id 및 actor_id가 포함된 `AgentCoreMemoryConfig`로 Memory 구성
- `AgentCoreMemorySessionManager` 인스턴스 생성
- session_manager를 사용하도록 Agent 설정

#### 3. Entry Point Handler
runtime_memory_agent 함수는 다음을 수행합니다.
- 입력 payload를 파싱하고 사용자 메시지 추출
- Agent 초기화 및 세션 추적 관리
- 적절한 컨텍스트로 Agent 호출 처리
- Runtime 환경에 형식이 지정된 응답 반환

Agent 파일을 생성합니다.

In [ ]:
%%writefile runtime_memory_agent.py
import os
import json
import logging
from typing import Dict, Any
from strands import Agent
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
# 상세 로깅 구성
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")

# AgentCore app 초기화
app = BedrockAgentCoreApp()

MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
REGION = os.getenv('AWS_REGION')

# 전역 Agent 인스턴스 - 첫 요청에서 초기화
agent = None
session_manager = None
 
   
    
def initialize_agent(actor_id, session_id):
    """메모리 세션 관리자로 에이전트를 초기화합니다."""
    global agent, session_manager

    logger.info(f"Initializing agent for actor_id={actor_id}, session_id={session_id}")

    # 모델 생성
    logger.info(f"Creating model with ID: {MODEL_ID}")
    model = BedrockModel(model_id=MODEL_ID)

    # AgentCoreMemoryConfig로 Memory 구성(권장)
    logger.info(f"Creating memory config with region: {REGION}")
    config = AgentCoreMemoryConfig(
        memory_id=MEMORY_ID,
        session_id=session_id,
        actor_id=actor_id
    )

    # Session Manager 생성 - 모든 메모리 작업을 자동으로 처리
    session_manager = AgentCoreMemorySessionManager(config, region_name=REGION)

    # session_manager를 사용하여 Agent 생성(사용자 지정 Hook 대체)
    logger.info("Creating agent with AgentCoreMemorySessionManager")
    agent = Agent(
        model=model,
        session_manager=session_manager,  # ✅ 내장 Memory 통합
        system_prompt="You're a helpful, memory-enabled agent deployed on AgentCore Runtime. You can remember previous interactions within the same session. Be friendly and concise in your responses."
    )
    logger.info("✅ Agent initialized with memory session manager")

@app.entrypoint
def runtime_memory_agent(payload, context):
    """
    메모리 지원 에이전트의 기본 진입점입니다.
    
    인자:
        payload: 사용자 데이터가 포함된 입력 페이로드
        context: 세션 정보가 포함된 Runtime 컨텍스트 객체
    """
    global agent, session_manager
    
    # payload와 context 정보를 모두 기록
    logger.info(f"Received payload: {payload}")
    logger.info(f"Context session_id: {context.session_id}")
    
    # 필수 값 추출 및 검증
    user_input = payload.get("prompt")
    actor_id = payload.get("actor_id", "default_user")  # 데모용 기본값 제공
    session_id = context.session_id  # context에서 session_id 가져오기
    
    # 필수 field 검증
    if user_input is None:
        error_msg = "❌ ERROR: Missing 'prompt' field in payload"
        logger.error(error_msg)
        return error_msg
    
    # 첫 요청에서 Agent 초기화
    if agent is None:
        logger.info("First request - initializing agent")
        initialize_agent(actor_id, session_id)
    else:
        
        # session 또는 actor 변경 여부 확인 - 변경되면 재초기화
        current_session = getattr(session_manager, '_session_id', None) if session_manager else None
        if current_session != session_id:
            logger.info(f"Session changed from {current_session} to {session_id} - reinitializing agent")
            initialize_agent(actor_id, session_id)
    
    # 사용자 입력으로 Agent 호출
    logger.info(f"Invoking agent with input: {user_input}")
    response = agent(user_input)
    response_text = response.message['content'][0]['text']
    logger.info(f"✅ Agent response: {response_text[:50]}...")
    
    return response_text

if __name__ == "__main__":
    logger.info("Starting AgentCore application")
    app.run()

## 3. AgentCore Runtime에 배포

이 섹션에서는 확장성과 간소화된 운영을 제공하는 관리형 Agent runtime 환경인 Amazon Bedrock AgentCore Runtime에 Agent를 배포합니다. AgentCore Runtime이 복잡한 인프라를 처리하므로 배포가 아니라 Agent logic에 집중할 수 있습니다.

수동 서버 설정과 관리가 필요한 기존 배포 방식과 달리 AgentCore Runtime은 코드를 container로 자동 packaging하여 AWS 인프라에 배포하고 호출용 보안 HTTPS endpoint를 제공합니다. 이 접근 방식은 Agent가 수요에 맞춰 확장되고 프로덕션 환경에서 안정적으로 작동하도록 보장합니다.

### 알아야 할 사항

- **AgentCore Runtime**은 Agent를 Docker container로 packaging하여 관리형 AWS 인프라에 배포합니다.
- **환경 변수**는 Agent를 구성합니다.
- `MEMORY_ID`: 앞서 생성한 Memory 리소스
- `MODEL_ID`: Claude Haiku 4.5 모델 ID
- `AWS_REGION`: 배포할 AWS 리전

> 💡 **팁**: AgentCore starter toolkit은 IAM 역할, ECR repository, container build를 포함한 복잡한 배포 단계를 모두 처리합니다.

### 배포 구성

배포 구성을 설정합니다.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import time

agentcore_runtime = Runtime()
agent_name = f"runtime_memory_agent_{unique_id}"

response = agentcore_runtime.configure(
    entrypoint="runtime_memory_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
    memory_mode="STM_ONLY",
)
response


### Memory 리소스 등록

1단계에서는 AgentCore Memory의 기반 구성 요소를 이해하기 위해 Memory 리소스를 수동으로 생성했습니다. 자체 Memory 리소스를 관리하므로 배포 시 이를 사용하도록 starter toolkit의 로컬 구성을 업데이트해야 합니다. 이렇게 하면 toolkit이 실행 시 새 Memory 리소스를 provision하지 않습니다.

다음 셀은 `configure()`에서 생성한 `.bedrock_agentcore.yaml` 구성 파일에 Memory 리소스 ID를 기록합니다.

> **참고**: AgentCore Starter Toolkit을 사용할 때는 Memory 리소스를 수동으로 관리할 필요가 없습니다. `configure()`에서 `memory_mode`를 설정하면 AgentCore Starter Toolkit이 Memory provision, 구성, 수명 주기 관리를 자동으로 처리합니다. 이 튜토리얼에서는 각 구성 요소의 end-to-end 작동 방식을 확인할 수 있도록 리소스를 명시적으로 생성합니다.

In [ ]:
import yaml

config_path = ".bedrock_agentcore.yaml"
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# 기존 memory_id를 Agent config에 주입
config["agents"][agent_name]["memory"]["memory_id"] = memory_id

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"✅ Injected memory_id: {memory_id} into {agent_name} config")

### Agent 실행

이제 Agent를 AgentCore Runtime에 실행합니다. 이 단계에서는 구성된 Agent를 AgentCore의 관리형 인프라에 배포합니다. 이 과정에서 앞서 생성한 Memory ID와 사용할 모델 ID 등 Agent에 필요한 필수 환경 변수도 전달합니다. 배포 후에는 사용자 메시지로 호출할 수 있는 보안 endpoint를 통해 Agent에 액세스할 수 있습니다.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "MEMORY_ID": memory_id,
        "MODEL_ID": "global.anthropic.claude-haiku-4-5-20251001-v1:0",
    }
)

### 배포 상태 확인

Agent의 배포 상태를 확인합니다.

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(f"Current status: {status}")

if status == "READY":
    print("✅ Agent successfully deployed!")
else:
    print(f"❌ Deployment ended with status: {status}")

## Agent 테스트

Agent가 배포되었으므로 메시지를 전송하여 테스트합니다.

**Actor ID 및 세션 관리에 관한 중요 참고 사항**

- Actor ID: 프로덕션 애플리케이션에서는 일반적으로 사용자가 로그인할 때 인증 시스템에서 actor_id를 가져옵니다. 이 식별자를 사용하면 Agent가 사용자별 대화 기록을 분리하여 유지할 수 있습니다. 이 예제에서는 하드 코딩된 값(test_user_123)을 사용하지만 실제 시나리오에서는 인증된 사용자의 고유 식별자를 전달합니다.

- 세션 관리: Session ID를 제공하지 않으면 AgentCore Runtime이 자동으로 생성하지만, 애플리케이션에서 Session ID를 명시적으로 관리하는 것이 좋습니다. 이를 통해 다음 항목을 더 효과적으로 제어할 수 있습니다.
- 세션 timeout 후 대화 계속
- 적절한 시점에 새 세션 생성(예: 사용자가 새 대화를 시작)
- 동일한 사용자의 여러 병렬 대화 처리
- 애플리케이션 요구 사항에 따른 세션 만료 policy 구현


In [ ]:
# 테스트 Session ID 생성
test_session_id = "agent-runtime-memory-session-123456789"  # 최소 길이는 33

# 첫 번째 메시지 전송
invoke_response = agentcore_runtime.invoke(
    {"prompt": "Hello! My name is John. What can you do?", "actor_id": "test_user_123"},
    session_id=test_session_id,
)

invoke_response

### Agent 응답 표시

응답을 더 읽기 쉬운 형식으로 표시합니다.

In [ ]:
from IPython.display import Markdown, display

response_text = invoke_response["response"][0]
display(Markdown(response_text))

### 지속성 테스트

동일한 세션에서 후속 메시지를 전송하여 Agent가 이전 상호작용을 기억하는지 테스트합니다.

In [ ]:
# 동일한 Session ID로 후속 메시지 전송
follow_up_response = agentcore_runtime.invoke(
    {"prompt": "What is my name?", "actor_id": "test_user_123"},
    session_id=test_session_id,
)

# 응답 표시
follow_up_text = follow_up_response["response"][0]
display(Markdown(follow_up_text))

### Memory 내용 검증

메시지가 올바르게 저장되었는지 확인하기 위해 Memory에 저장된 내용을 살펴봅니다.

In [ ]:
# 세션 작업에 MemorySessionManager 사용
manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)
session = manager.create_memory_session(actor_id="test_user_123", session_id=test_session_id)

# 세션 범위 method로 대화 기록 가져오기
stored_turns = session.get_last_k_turns(k=10)

print(f"Found {len(stored_turns)} conversation turns in memory (shown in chronological order):")
for idx, turn in enumerate(stored_turns):
    print(f"\nTurn {idx + 1}:")
    for message in turn:
        role = message["role"]
        text = message["content"]["text"]
        print(f"- {role}: {text[:100]}...")

## 핵심 개념

1. **Memory 통합**: Amazon Bedrock Memory를 사용하여 대화 기록을 저장하는 방법
2. **세션 관리**: Session ID를 사용하여 대화 컨텍스트를 유지하는 방법
3. **AgentCore 배포**: Agent를 프로덕션 Runtime 환경에 배포하는 방법
4. **AgentCoreMemorySessionManager**: 자동 Memory 처리를 위한 기본 제공 Strands 통합 기능을 사용하는 방법

이러한 개념은 지속형 Memory를 갖춘 더 복잡한 Agent를 구축하는 기반이 됩니다.

## 리소스 정리(선택 사항)

이 튜토리얼에서 생성한 리소스가 더 이상 필요하지 않으면 정리할 수 있습니다.

In [ ]:
# 리소스 식별자 가져오기
if "launch_result" in locals():
    print(f"Agent ID: {launch_result.agent_id}")
    print(f"ECR Repository: {launch_result.ecr_uri.split('/')[1]}")
else:
    print("Launch results not available")

In [ ]:
# 리소스를 삭제하려는 경우에만 이 셀 실행

# AgentCore Runtime 삭제
agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)
print(f"Deleted AgentCore Runtime: {launch_result.agent_id}")

# ECR repository 삭제
ecr_client = boto3.client("ecr", region_name=REGION)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1].split(":")[0], force=True)
print(f"Deleted ECR repository: {launch_result.ecr_uri.split('/')[1].split(':')[0]}")

# Memory 리소스 삭제
memory_client = MemoryClient(region_name=REGION)
memory_client.delete_memory_and_wait(memory_id=memory_id)
print(f"Deleted memory resource: {memory_id}")

## 축하합니다!

Amazon Bedrock AgentCore Runtime과 AgentCore Memory를 사용하여 첫 번째 Memory 지원 Agent를 성공적으로 구축하고 배포했습니다.

### 다음 단계

기본 사항을 이해했으므로 다음 내용을 진행할 수 있습니다.

1. **도구 추가**: 계산기, database connector 또는 API 호출 같은 도구로 Agent 개선
2. **Memory 개선**: 장기 메모리를 사용하는 더 정교한 Memory strategy 구현
3. **UI 구축**: Agent용 web 또는 mobile interface 생성